In [5]:
import sys, time
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from pyspark.sql import functions as F, Window
from src.config import load_config
from src.spark_session import get_spark          # ajusta al nombre de tu función
from src.models.popularity import add_recent_counts, smoothed_ctr
from src.models.embeddings import user_profiles_dense, score_dense

cfg = load_config("small")
OUT = cfg["paths"]["processed"]
spark = get_spark(cfg)

PERIODO_TEST = "test"
W = 0.4

t0 = time.time()
imp = spark.read.parquet(str(OUT / "impressions_long"))
beh = spark.read.parquet(str(OUT / "behaviors"))
news_emb = spark.read.parquet(str(OUT / "news_emb_minilm")).cache()

# Señal 1: CTR en streaming (24 h, alpha = 20), p0 con 9–14 nov
p0 = imp.filter(F.col("periodo") != PERIODO_TEST).agg(F.avg("label")).first()[0]
base = (imp.filter(F.col("time") >= F.to_timestamp(F.lit("2019-11-14 00:00:00")))
           .select("impression_id", "user_id", "time", "news_id", "label", "periodo"))
cands = (add_recent_counts(base, hours=(24,))
         .filter(F.col("periodo") == PERIODO_TEST)
         .withColumn("ctr", smoothed_ctr(24, 20, p0))
         .select("impression_id", "user_id", "news_id", "label", "ctr"))

# Señal 2: coseno con embeddings (historial completo)
users_test = (beh.filter(F.col("periodo") == PERIODO_TEST)
                 .select("user_id", "history").dropDuplicates(["user_id"]))
profiles = user_profiles_dense(users_test, news_emb, k=None)

# Puntaje z dentro de la impresión (0 si la desviación es 0 o nula)
w_imp = Window.partitionBy("impression_id")
def zscore(col):
    mu, sd = F.avg(col).over(w_imp), F.stddev_pop(col).over(w_imp)
    return F.when((sd > 0) & F.col(col).isNotNull(), (F.col(col) - mu) / sd).otherwise(F.lit(0.0))

senales = (score_dense(cands, news_emb, profiles)
           .withColumn("s_base", F.col("ctr"))
           .withColumn("s_hib", W * zscore("cos") + (1 - W) * zscore("ctr"))
           .select("impression_id", "user_id", "news_id", "label", "ctr", "cos", "s_base", "s_hib"))

pdf = senales.toPandas()
print(f"Candidatas: {len(pdf):,} | impresiones: {pdf.impression_id.nunique():,} ({time.time() - t0:.0f} s)")

Candidatas: 2,740,998 | impresiones: 73,152 (668 s)


In [6]:
rng = np.random.default_rng(42)
pdf["desempate"] = rng.random(len(pdf))        # empates al azar con semilla, como en la evaluación

def auc_impresion(y, s):
    """AUC = P(puntaje de un clic > puntaje de un no clic); empates cuentan 0,5."""
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    dif = pos[:, None] - neg[None, :]
    return ((dif > 0).sum() + 0.5 * (dif == 0).sum()) / dif.size

filas = []
for imp_id, g in pdf.groupby("impression_id", sort=False):
    y = g["label"].to_numpy()
    filas.append((imp_id, g["user_id"].iat[0], len(g), y.sum(),
                  auc_impresion(y, g["s_base"].to_numpy()),
                  auc_impresion(y, g["s_hib"].to_numpy())))
por_imp = pd.DataFrame(filas, columns=["impression_id", "user_id", "n_cand", "n_clics",
                                       "auc_base", "auc_hib"])
por_imp["delta"] = por_imp["auc_hib"] - por_imp["auc_base"]

# Verificación: debe reproducir ~0,6526 y ~0,6832
print(f"AUC baseline = {por_imp.auc_base.mean():.4f} | híbrido = {por_imp.auc_hib.mean():.4f}")
print(f"Impresiones donde el híbrido mejora: {(por_imp.delta > 0).mean():.1%} | "
      f"empeora: {(por_imp.delta < 0).mean():.1%} | igual: {(por_imp.delta == 0).mean():.1%}")

AUC baseline = 0.6526 | híbrido = 0.6832
Impresiones donde el híbrido mejora: 43.0% | empeora: 28.2% | igual: 28.8%


In [7]:
N_MEJOR, N_PEOR = 3, 2
COLS_NEWS = ["news_id", "category", "title"]
news = spark.read.parquet(str(OUT / "news")).select(*COLS_NEWS).toPandas().set_index("news_id")
hist = (users_test.toPandas().set_index("user_id")["history"])

# Casos legibles: un solo clic, entre 10 y 40 candidatas, historial de al menos 5 noticias
legibles = por_imp[(por_imp.n_clics == 1) & por_imp.n_cand.between(10, 40)
                   & por_imp.user_id.map(lambda u: len(hist.get(u, []) or []) >= 5)]

# Se muestrea entre el 5 % extremo, para no quedarse solo con los casos más atípicos
mejores = legibles[legibles.delta >= legibles.delta.quantile(0.95)].sample(N_MEJOR, random_state=42)
peores  = legibles[legibles.delta <= legibles.delta.quantile(0.05)].sample(N_PEOR,  random_state=42)

def rangos(g, col):
    orden = g.sort_values([col, "desempate"], ascending=[False, False])
    return pd.Series(np.arange(1, len(g) + 1), index=orden.index)

def mostrar(caso, etiqueta):
    g = pdf[pdf.impression_id == caso.impression_id].copy()
    g["rango_base"], g["rango_hib"] = rangos(g, "s_base"), rangos(g, "s_hib")
    clic = g[g.label == 1].iloc[0]
    h = hist[caso.user_id]
    print(f"\n{'=' * 90}\n{etiqueta} | impresión {caso.impression_id} | usuario {caso.user_id} | "
          f"{caso.n_cand} candidatas | AUC base {caso.auc_base:.2f} -> híbrido {caso.auc_hib:.2f}")
    print(f"Historial ({len(h)} noticias), últimas 5:")
    for n in h[-5:]:
        print(f"   [{news.at[n, 'category']}] {news.at[n, 'title']}")
    print(f"Noticia cliqueada: [{news.at[clic.news_id, 'category']}] {news.at[clic.news_id, 'title']}")
    print(f"   rango baseline = {clic.rango_base} | rango híbrido = {clic.rango_hib} | "
          f"CTR = {clic.ctr:.3f} | coseno = {clic.cos:.3f}")
    print("Top 3 del baseline:")
    for _, r in g.nsmallest(3, "rango_base").iterrows():
        print(f"   {r.rango_base}. [{news.at[r.news_id, 'category']}] {news.at[r.news_id, 'title']} "
              f"(CTR {r.ctr:.3f}, cos {r.cos:.3f}, clic={r.label})")
    return {"tipo": etiqueta, "impression_id": caso.impression_id, "n_cand": caso.n_cand,
            "titulo_clic": news.at[clic.news_id, "title"], "categoria_clic": news.at[clic.news_id, "category"],
            "rango_base": clic.rango_base, "rango_hib": clic.rango_hib,
            "ctr": round(clic.ctr, 3), "cos": round(clic.cos, 3)}

casos = [mostrar(c, "MEJORA") for c in mejores.itertuples()] + \
        [mostrar(c, "EMPEORA") for c in peores.itertuples()]

pd.DataFrame(casos).to_csv(ROOT / "results" / "metrics" / "10_casos_errores.csv", index=False)


MEJORA | impresión 44667 | usuario U56522 | 19 candidatas | AUC base 0.22 -> híbrido 0.83
Historial (15 noticias), últimas 5:
   [health] Gradual lifestyle changes helped this woman lose 75 pounds and keep it off
   [entertainment] 4 Things to Know About Keanu Reeves' New Girlfriend
   [tv] 50 Celebrities You Definitely Forgot Busted a Move on 'Dancing With the Stars'
   [lifestyle] 100 Incredible Christmas Tree Decorating Ideas
   [lifestyle] 30 style shortcuts for no-hassle Christmas decor
Noticia cliqueada: [entertainment] THEN AND NOW: What all your favorite '90s stars are doing today
   rango baseline = 15 | rango híbrido = 4 | CTR = 0.031 | coseno = 0.444
Top 3 del baseline:
   1. [sports] Opinion: Colin Kaepernick is about to get what he deserves: a chance (CTR 0.184, cos 0.123, clic=0)
   2. [news] South Carolina teen gets life in prison for deadly elementary school shooting (CTR 0.133, cos 0.078, clic=0)
   3. [lifestyle] Meghan Markle and Hillary Clinton Secretly Spent the A